In [ ]:
import re
import os
import unicodedata
import urllib3
import zipfile
import shutil
import numpy as np
import pandas as pd
import torch
from collections import Counter
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
num_samples = 33000

In [ ]:
!wget -c http://www.manythings.org/anki/fra-eng.zip && unzip -o fra-eng.zip

In [ ]:
def unicode_to_ascii(s):
  # Remove French accents
  # Example: 'déjà diné' -> deja dine
  return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [ ]:
def preprocess_sentence(sent):
  # Call the accent-removal function
  sent = unicode_to_ascii(sent.lower())

  # Insert spaces between words and punctuation.
  # Ex) "he is a boy." => "he is a boy ."
  sent = re.sub(r"([?.!,¿])", r" \1", sent)

  # Replace every character except (a-z, A-Z, '.', '?', '!', ',') with a space.
  sent = re.sub(r"[^a-zA-Z!.?]+", r" ", sent)

  # Replace multiple spaces with a single space
  sent = re.sub(r"\s+", " ", sent)
  return sent

In [ ]:
def load_preprocessed_data():
  encoder_input, decoder_input, decoder_target = [], [], []

  with open("fra.txt", "r") as lines:
    for i, line in enumerate(lines):
      # Separate the source data and target data
      src_line, tar_line, _ = line.strip().split('\t')

      # Preprocess the source data
      src_line = [w for w in preprocess_sentence(src_line).split()]

      # Preprocess the target data
      tar_line = preprocess_sentence(tar_line)
      tar_line_in = [w for w in ("<sos> " + tar_line).split()]
      tar_line_out = [w for w in (tar_line + " <eos>").split()]

      encoder_input.append(src_line)
      decoder_input.append(tar_line_in)
      decoder_target.append(tar_line_out)

      if i == num_samples - 1:
        break

  return encoder_input, decoder_input, decoder_target

In [ ]:
# Test preprocessing
en_sent = u"Have you had dinner?"
fr_sent = u"Avez-vous déjà diné?"

print('English sentence before preprocessing:', en_sent)
print('English sentence after preprocessing:',preprocess_sentence(en_sent))
print('French sentence before preprocessing:', fr_sent)
print('French sentence after preprocessing:', preprocess_sentence(fr_sent))

In [ ]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()

In [ ]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()
print('Encoder input:',sents_en_in[:5])
print('Decoder input:',sents_fra_in[:5])
print('Decoder labels:',sents_fra_out[:5])

In [ ]:
def build_vocab(sents):
  word_list = []

  for sent in sents:
      for word in sent:
        word_list.append(word)

  # Calculate the frequency of each word and sort by descending frequency
  word_counts = Counter(word_list)
  vocab = sorted(word_counts, key=word_counts.get, reverse=True)

  word_to_index = {}
  word_to_index['<PAD>'] = 0
  word_to_index['<UNK>'] = 1

  # Assign smaller integer IDs to words with higher frequencies
  for index, word in enumerate(vocab) :
    word_to_index[word] = index + 2

  return word_to_index

In [ ]:
src_vocab = build_vocab(sents_en_in)
tar_vocab = build_vocab(sents_fra_in + sents_fra_out)

src_vocab_size = len(src_vocab)
tar_vocab_size = len(tar_vocab)
print("English vocabulary size: {:d}, French vocabulary size: {:d}".format(src_vocab_size, tar_vocab_size))

In [ ]:
index_to_src = {v: k for k, v in src_vocab.items()}
index_to_tar = {v: k for k, v in tar_vocab.items()}

def texts_to_sequences(sents, word_to_index):
  encoded_X_data = []
  for sent in tqdm(sents):
    index_sequences = []
    for word in sent:
      try:
          index_sequences.append(word_to_index[word])
      except KeyError:
          index_sequences.append(word_to_index['<UNK>'])
    encoded_X_data.append(index_sequences)
  return encoded_X_data

In [ ]:
encoder_input = texts_to_sequences(sents_en_in, src_vocab)
decoder_input = texts_to_sequences(sents_fra_in, tar_vocab)
decoder_target = texts_to_sequences(sents_fra_out, tar_vocab)

In [ ]:
# Print the sentences before and after integer encoding for the first five samples
# Since this is encoder input, there is no <sos> or <eos>
for i, (item1, item2) in zip(range(5), zip(sents_en_in, encoder_input)):
    print(f"Index: {i}, Before integer encoding: {item1}, After integer encoding: {item2}")

In [ ]:
def pad_sequences(sentences, max_len=None):
    # If max_len is not given, pad to the maximum length in the data
    if max_len is None:
        max_len = max([len(sentence) for sentence in sentences])

    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index, :len(sentence)] = np.array(sentence)[:max_len]
    return features

In [ ]:
encoder_input = pad_sequences(encoder_input)
decoder_input = pad_sequences(decoder_input)
decoder_target = pad_sequences(decoder_target)

In [ ]:
print('Shape of the encoder input:',encoder_input.shape)
print('Shape of the decoder input:',decoder_input.shape)
print('Shape of the decoder labels:',decoder_target.shape)

In [ ]:
indices = np.arange(encoder_input.shape[0])
np.random.shuffle(indices)
print('Random sequence:',indices)

In [ ]:
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

In [ ]:
print([index_to_src[word] for word in encoder_input[30997]])
print([index_to_tar[word] for word in decoder_input[30997]])
print([index_to_tar[word] for word in decoder_target[30997]])

In [ ]:
n_of_val = int(33000*0.1)
print('Number of validation samples:',n_of_val)

In [ ]:
encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

In [ ]:
print('Shape of the training source data:',encoder_input_train.shape)
print('Shape of the training target data:',decoder_input_train.shape)
print('Shape of the training target labels:',decoder_target_train.shape)
print('Shape of the test source data:',encoder_input_test.shape)
print('Shape of the test target data:',decoder_input_test.shape)
print('Shape of the test target labels:',decoder_target_test.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

embedding_dim = 256
hidden_units = 256

class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embedding_dim, hidden_units):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(src_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_units, batch_first=True)

    def forward(self, x):
        # x.shape == (batch_size, seq_len, embedding_dim)
        x = self.embedding(x)
        # hidden.shape == (1, batch_size, hidden_units), cell.shape == (1, batch_size, hidden_units)
        _, (hidden, cell) = self.lstm(x)
        # The encoder outputs the hidden state and cell state
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, tar_vocab_size, embedding_dim, hidden_units):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(tar_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_units, batch_first=True)
        self.fc = nn.Linear(hidden_units, tar_vocab_size)

    def forward(self, x, hidden, cell):

        # x.shape == (batch_size, seq_len, embedding_dim)
        x = self.embedding(x)

        # Pass the encoder's hidden state and cell state to the decoder LSTM.
        # output.shape == (batch_size, seq_len, hidden_units)
        # hidden.shape == (1, batch_size, hidden_units)
        # cell.shape == (1, batch_size, hidden_units)
        output, (hidden, cell) = self.lstm(x, (hidden, cell))

        # output.shape: (batch_size, seq_len, tar_vocab_size)
        output = self.fc(output)

        # The decoder outputs the prediction, hidden state, and cell state
        return output, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden, cell = self.encoder(src)

        # During training, only output among the decoder outputs is used.
        output, _, _ = self.decoder(trg, hidden, cell)
        return output

encoder = Encoder(src_vocab_size, embedding_dim, hidden_units)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_units)
model = Seq2Seq(encoder, decoder)

loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters())

In [ ]:
print(model)

In [ ]:
def evaluation(model, dataloader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for encoder_inputs, decoder_inputs, decoder_targets in dataloader:
            encoder_inputs = encoder_inputs.to(device)
            decoder_inputs = decoder_inputs.to(device)
            decoder_targets = decoder_targets.to(device)

            # Forward propagation
            # outputs.shape == (batch_size, seq_len, tar_vocab_size)
            outputs = model(encoder_inputs, decoder_inputs)

            # Compute the loss
            # Shape of outputs.view(-1, outputs.size(-1)) is (batch_size * seq_len, tar_vocab_size)
            # Shape of decoder_targets.view(-1) is (batch_size * seq_len)
            loss = loss_function(outputs.view(-1, outputs.size(-1)), decoder_targets.view(-1))
            total_loss += loss.item()

            # Compute accuracy (excluding padding tokens)
            mask = decoder_targets != 0
            total_correct += ((outputs.argmax(dim=-1) == decoder_targets) * mask).sum().item()
            total_count += mask.sum().item()

    return total_loss / len(dataloader), total_correct / total_count

In [ ]:
encoder_input_train_tensor = torch.tensor(encoder_input_train, dtype=torch.long)
decoder_input_train_tensor = torch.tensor(decoder_input_train, dtype=torch.long)
decoder_target_train_tensor = torch.tensor(decoder_target_train, dtype=torch.long)

encoder_input_test_tensor = torch.tensor(encoder_input_test, dtype=torch.long)
decoder_input_test_tensor = torch.tensor(decoder_input_test, dtype=torch.long)
decoder_target_test_tensor = torch.tensor(decoder_target_test, dtype=torch.long)

# Create the dataset and dataloader
batch_size = 128

train_dataset = TensorDataset(encoder_input_train_tensor, decoder_input_train_tensor, decoder_target_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = TensorDataset(encoder_input_test_tensor, decoder_input_test_tensor, decoder_target_test_tensor)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Training settings
num_epochs = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Training loop
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # Training mode
    model.train()

    for encoder_inputs, decoder_inputs, decoder_targets in train_dataloader:
        encoder_inputs = encoder_inputs.to(device)
        decoder_inputs = decoder_inputs.to(device)
        decoder_targets = decoder_targets.to(device)

        # Initialize gradients
        optimizer.zero_grad()

        # Forward propagation
        # outputs.shape == (batch_size, seq_len, tar_vocab_size)
        outputs = model(encoder_inputs, decoder_inputs)

        # Compute the loss and perform backpropagation
        # Shape of outputs.view(-1, outputs.size(-1)) is (batch_size * seq_len, tar_vocab_size)
        # Shape of decoder_targets.view(-1) is (batch_size * seq_len)
        loss = loss_function(outputs.view(-1, outputs.size(-1)), decoder_targets.view(-1))
        loss.backward()

        # Update the weights
        optimizer.step()

    train_loss, train_acc = evaluation(model, train_dataloader, loss_function, device)
    valid_loss, valid_acc = evaluation(model, valid_dataloader, loss_function, device)

    print(f'Epoch: {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}')

    # Save a checkpoint when the validation loss is minimized
    if valid_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {valid_loss:.4f}. Saving the checkpoint.')
        best_val_loss = valid_loss
        torch.save(model.state_dict(), 'best_model_checkpoint.pth')

In [ ]:
# Load the model
model.load_state_dict(torch.load('best_model_checkpoint.pth'))

# Move the model to the device
model.to(device)

# Compute accuracy and loss on the validation data
val_loss, val_accuracy = evaluation(model, valid_dataloader, loss_function, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

In [ ]:
print(tar_vocab['<sos>'])
print(tar_vocab['<eos>'])

In [ ]:
index_to_src = {v: k for k, v in src_vocab.items()}
index_to_tar = {v: k for k, v in tar_vocab.items()}

# Convert the source integer sequence into a text sequence
def seq_to_src(input_seq):
  sentence = ''
  for encoded_word in input_seq:
    if(encoded_word != 0):
      sentence = sentence + index_to_src[encoded_word] + ' '
  return sentence

# Convert the translated integer sequence into a text sequence
def seq_to_tar(input_seq):
  sentence = ''
  for encoded_word in input_seq:
    if(encoded_word != 0 and encoded_word != tar_vocab['<sos>'] and encoded_word != tar_vocab['<eos>']):
      sentence = sentence + index_to_tar[encoded_word] + ' '
  return sentence

In [ ]:
def decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, max_output_len, int_to_src_token, int_to_tar_token):
    encoder_inputs = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)

    # Set the encoder's initial state
    hidden, cell = model.encoder(encoder_inputs)

    # Use the start token <sos> as the first decoder input
    # unsqueeze(0) adds the batch dimension.
    decoder_input = torch.tensor([3], dtype=torch.long).unsqueeze(0).to(device)

    decoded_tokens = []

    # Each iteration of the loop corresponds to one decoder time step
    for _ in range(max_output_len):
        output, hidden, cell = model.decoder(decoder_input, hidden, cell)

        # Perform softmax regression and obtain the predicted word index
        output_token = output.argmax(dim=-1).item()

        # End token <eos>
        if output_token == 4:
            break

        # Accumulate the word ID at each time step in decoded_tokens and return the final translated sequence.
        decoded_tokens.append(output_token)

        # Prediction at the current time step. It is used as the input for the next time step.
        decoder_input = torch.tensor([output_token], dtype=torch.long).unsqueeze(0).to(device)

    return ' '.join(int_to_tar_token[token] for token in decoded_tokens)

In [ ]:
for seq_index in [3, 50, 100, 300, 1001]:
  input_seq = encoder_input_train[seq_index]
  translated_text = decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, 20, index_to_src, index_to_tar)

  print("Input sentence:",seq_to_src(encoder_input_train[seq_index]))
  print("Reference sentence:",seq_to_tar(decoder_input_train[seq_index]))
  print("Translated sentence:",translated_text)
  print("-"*50)

In [ ]:
for seq_index in [3, 50, 100, 300, 1001]:
  input_seq = encoder_input_test[seq_index]
  translated_text = decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, 20, index_to_src, index_to_tar)

  print("Input sentence:",seq_to_src(encoder_input_test[seq_index]))
  print("Reference sentence:",seq_to_tar(decoder_input_test[seq_index]))
  print("Translated sentence:",translated_text)
  print("-"*50)